# An introduction to plotlet

plotlet is a Python library for SVG plots — reproducible, byte-identical, with a standard plotting vocabulary built around the data-frame-first idiom.

This notebook walks through the way plotlet wants you to think about a figure: **bind a data frame to a chart, layer one or more marks on top, compose multi-panel layouts**. We'll use the bundled Palmer Penguins dataset throughout.

In [8]:
import pandas as pd
import plotlet as pt

# pt.load returns a column-oriented dict; wrapping it in a DataFrame is
# optional but gives a nice tabular preview and any pandas operations you want.
penguins = pd.DataFrame(pt.load("penguins"))
penguins.head()

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex,year
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,male,2007
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,female,2007
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,female,2007
3,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN,2007
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,female,2007


## The basic pattern

Pass a data frame to `pt.chart(...)` and refer to its columns by name in every mark call. `color=` (or `fill=` for filled artists) maps a categorical column to one tab10 color per level. Mark methods return the chart, so they chain.

In [9]:
c = pt.chart(penguins,
             title="bill dimensions by species",
             xlabel="bill length (mm)", ylabel="bill depth (mm)",
             legend=True)
c.scatter(x="bill_length_mm", y="bill_depth_mm", color="species",
          s=22, alpha=0.7)
c.legend(position="right")
c

## Declare aesthetics once, layer many marks

Bind `data`, `x`, `y`, `color` at the chart level and every subsequent mark inherits them. Per-call kwargs always win — only unset slots fall back to the chart-level value.

The canonical example is a boxplot with raw points overlaid, both using the same `x` and `y`:

In [10]:
c = pt.chart(penguins, x="species", y="body_mass_g",
             title="body mass by species (boxplot + strip)",
             ylabel="body mass (g)", data_height=260)
c.boxplot()
c.strip(s=3, alpha=0.5)
c

Both `boxplot()` and `strip()` are called with **no arguments** — they inherit `x="species"` and `y="body_mass_g"` from the chart. The same aes apply to every layer; if you want to override one, pass it explicitly:

```python
c.scatter(x="other_col")   # overrides x for just this call
```

## Distributions across groups

`density_1d` (Gaussian KDE) plus `rug` (raw observations as tick marks) makes a clean overlay. Sharing `color=` at the chart level keeps the colors consistent across both layers.

In [11]:
c = pt.chart(penguins, x="flipper_length_mm", color="species",
             title="flipper length, by species",
             xlabel="flipper length (mm)", ylabel="density",
             legend=True)
c.density_1d(fill=True)
c.rug()
c.legend(position="right")
c

## Multi-panel composition

The `|` operator stacks charts side-by-side; `/` stacks them vertically. They nest — use `(a | b) / c` for an L-shape, or `pt.grid([[a, b], [c, d]])` for arbitrary 2-D layouts. The composed chart owns its children — render the parent.

Three views of body mass in one figure:

In [12]:
violin = pt.chart(penguins, x="species", y="body_mass_g",
                  title="violin", ylabel="body mass (g)",
                  data_width=260, data_height=200)
violin.violin()

swarm = pt.chart(penguins, x="species", y="body_mass_g",
                 title="swarm", data_width=260, data_height=200)
swarm.swarm(size=3)

density = pt.chart(penguins, x="body_mass_g", color="species",
                   title="density", xlabel="body mass (g)",
                   ylabel="density", data_height=140, legend=True)
density.density_1d(fill=True)
density.legend(position="right")

(violin | swarm) / density

## Shared scales

`(a / b).share_x()` collapses the gap between two panels and unions their x-domains so the scale is identical across them. The first leaf is the anchor. The classic use case is a scatter with a marginal distribution below — the x-axis is locked so the marginal density tracks exactly with the data above.

In [13]:
top = pt.chart(penguins, x="bill_length_mm", y="body_mass_g", color="species",
               title="scatter + marginal density",
               ylabel="body mass (g)", data_height=200, legend=True)
top.scatter(s=18, alpha=0.6)
top.legend(position="right")

bot = pt.chart(penguins, x="bill_length_mm", color="species",
               xlabel="bill length (mm)", ylabel="density",
               data_height=90)
bot.density_1d(fill=True)

(top / bot).share_x()

## A taste of the AI-readable SVG

Every figure emits `data-plotlet-*` attributes describing plot type, axes, scales, ranges, and labels — enough for an AI or downstream tool to understand the figure from one XML parse. Schema: [`docs/AI_ATTRS.md`](../docs/AI_ATTRS.md).

In [14]:
svg = (pt.chart(penguins, x="bill_length_mm", y="bill_depth_mm", color="species")
         .scatter(s=20).to_svg())

import re
for m in re.finditer(r'data-plotlet-\w+="[^"]{0,40}"', svg[:1500]):
    print(m.group(0))

data-plotlet-version="0.4.5"
data-plotlet-schema="2"
data-plotlet-kind="figure"
data-plotlet-kind="panel"
data-plotlet-xscale="linear"
data-plotlet-yscale="linear"
data-plotlet-xlim="30.725,60.975"
data-plotlet-ylim="12.68,21.92"
data-plotlet-type="scatter"
data-plotlet-index="0"
data-plotlet-label="Adelie"
data-plotlet-color="#1f77b4"
data-plotlet-n="152"
data-plotlet-marker="o"


## Where to next

- [`docs/API.md`](../docs/API.md) — full method reference, frame options, scales
- [`docs/PHILOSOPHY.md`](../docs/PHILOSOPHY.md) — what's in core vs extensions and why
- [`docs/SUBPLOTS.md`](../docs/SUBPLOTS.md) — multi-panel composition in depth
- [`docs/EXTENDING.md`](../docs/EXTENDING.md) — write your own plot type (~50–100 lines)

The penguins dataset used here is from [Gorman, Williams & Fraser (2014)](https://allisonhorst.github.io/palmerpenguins/), released under CC0.